# Loss Masking for Reasoning Models

This notebook demonstrates `apply_mask()` from `thinkpack`, showing how different `MaskType` combinations affect which tokens contribute to the training loss.

**The think-collapse problem:** fine-tuning a thinking model on standard instruction/response data (with no special handling) causes the model to stop using `<think>` blocks, reducing reasoning capability.

**Two models are used throughout:**
- **Qwen3** — INLINE reasoning; the chat template always emits `<think>\n\n</think>` even when no reasoning is provided.
- **OLMo-3-Think** — PREFIXED reasoning; the template only appends `<think>` as a generation primer at inference time, so the think block is absent from training sequences unless explicitly included.

**Three masking modes:**
- `masked=None` — no masking; all tokens contribute to the loss (naive baseline).
- `masked=MaskType.THINK` — mask the think block only; train on instruction + response.
- `masked=MaskType.PROMPT | MaskType.THINK` — mask instruction and think block; train on response only.

In [1]:
import sys

# add the src directory to the path so thinkpack can be imported without installation
sys.path.insert(0, "../src")

from transformers import AutoTokenizer

from thinkpack import MaskType, apply_mask, detect_model

/Users/lukas.twist/code/thinkguard/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## 1. Load Tokenizers

We load tokenizers for both models. `detect_model()` inspects the chat template to determine how each model handles reasoning blocks — `prefixed=True` means the template injects `<think>` into the generation prompt; `prefixed=False` means the model generates it itself.

In [2]:
# load tokenizers for both models (cpu-only, no gpu needed)
tok_qwen = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B", trust_remote_code=True)
tok_olmo = AutoTokenizer.from_pretrained(
    "allenai/OLMo-3-7B-Think",
    trust_remote_code=True,
)

# detect model properties — prefixed indicates whether the template injects <think>
info_qwen = detect_model(tok_qwen)
info_olmo = detect_model(tok_olmo)

print(f"Qwen3  — prefixed: {info_qwen.prefixed}, tag: {info_qwen.open_tag}")
print(f"OLMo-3 — prefixed: {info_olmo.prefixed}, tag: {info_olmo.open_tag}")

Qwen3  — prefixed: False, tag: <think>
OLMo-3 — prefixed: True, tag: <think>


## 2. Using `apply_mask()`

`apply_mask()` takes a list of conversations and returns a HuggingFace `Dataset` with `input_ids`, `labels`, and `attention_mask`.

Each conversation is a list of message dicts — the same format as `apply_chat_template()`:

```python
[
    {"role": "user", "content": "What is 2+2?"},
    {"role": "assistant", "content": "4"},
]
```

The assistant message may include an optional `"reasoning"` key with the think block content. If absent when masking is active, an empty block is injected automatically so the training context matches inference time.

Tokens where `label == -100` are ignored by PyTorch's `CrossEntropyLoss` and all major training frameworks (Trainer, SFTTrainer, unsloth).

In [3]:
# helper to print each token alongside its training/masked status
def show_token_labels(dataset, tokenizer, label: str) -> None:
    """Print each token with its training status for the first record in a dataset."""
    input_ids = dataset[0]["input_ids"]
    labels = dataset[0]["labels"]
    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    n_masked = labels.count(-100)
    n_trainable = len(labels) - n_masked
    print(f"=== {label} ===")
    print(f"  total tokens: {len(input_ids)}, masked: {n_masked}, trainable: {n_trainable}")
    for tok_str, lbl in zip(tokens, labels):
        status = "MASK " if lbl == -100 else "train"
        print(f"  [{status}] {repr(tok_str)}")
    print()


# a minimal conversation — instruction only, no reasoning
demo_conversation = [
    {"role": "user", "content": "What is 2+2?"},
    {"role": "assistant", "content": "4"},
]

### 2a. No masking (`masked=None`) — naive baseline

All tokens contribute to the loss. For Qwen3 the template always emits an empty `<think>\n\n</think>` block — training on it directly teaches the model to produce empty thinking. For OLMo-3 the think block is absent entirely, so the training and inference contexts diverge. Both are examples of think collapse.

In [4]:
# masked=None — no sections masked, all tokens trainable
for name, tok in [("Qwen3", tok_qwen), ("OLMo-3", tok_olmo)]:
    ds = apply_mask(
        conversations=[demo_conversation],
        tokenizer=tok,
        masked=None,
    )
    show_token_labels(ds, tok, f"{name} — no masking (masked=None)")

=== Qwen3 — no masking (masked=None) ===
  total tokens: 21, masked: 0, trainable: 21
  [train] '<|im_start|>'
  [train] 'user'
  [train] 'Ċ'
  [train] 'What'
  [train] 'Ġis'
  [train] 'Ġ'
  [train] '2'
  [train] '+'
  [train] '2'
  [train] '?'
  [train] '<|im_end|>'
  [train] 'Ċ'
  [train] '<|im_start|>'
  [train] 'assistant'
  [train] 'Ċ'
  [train] '<think>'
  [train] 'ĊĊ'
  [train] '</think>'
  [train] 'ĊĊ'
  [train] '4'
  [train] '<|im_end|>'

=== OLMo-3 — no masking (masked=None) ===
  total tokens: 81, masked: 0, trainable: 81
  [train] '<|im_start|>'
  [train] 'system'
  [train] 'Ċ'
  [train] 'You'
  [train] 'Ġare'
  [train] 'ĠO'
  [train] 'LM'
  [train] 'o'
  [train] ','
  [train] 'Ġa'
  [train] 'Ġhelpful'
  [train] 'Ġfunction'
  [train] '-c'
  [train] 'alling'
  [train] 'ĠAI'
  [train] 'Ġassistant'
  [train] 'Ġbuilt'
  [train] 'Ġby'
  [train] 'ĠAi'
  [train] '2'
  [train] '.'
  [train] 'ĠYour'
  [train] 'Ġdate'
  [train] 'Ġcutoff'
  [train] 'Ġis'
  [train] 'ĠNovember'
  [train

### 2b. Mask think block only (`masked=MaskType.THINK`)

The think block is included in the training sequence so training/inference context matches, but its tokens are masked so no gradient flows through them. The model trains on instruction and response only.

For conversations without a `"reasoning"` key, an empty think block is automatically injected.

In [5]:
# masked=MaskType.THINK — think block masked, instruction and response trainable
for name, tok in [("Qwen3", tok_qwen), ("OLMo-3", tok_olmo)]:
    ds = apply_mask(
        conversations=[demo_conversation],
        tokenizer=tok,
        masked=MaskType.THINK,
    )
    show_token_labels(ds, tok, f"{name} — think masked (MaskType.THINK)")

=== Qwen3 — think masked (MaskType.THINK) ===
  total tokens: 21, masked: 4, trainable: 17
  [train] '<|im_start|>'
  [train] 'user'
  [train] 'Ċ'
  [train] 'What'
  [train] 'Ġis'
  [train] 'Ġ'
  [train] '2'
  [train] '+'
  [train] '2'
  [train] '?'
  [train] '<|im_end|>'
  [train] 'Ċ'
  [train] '<|im_start|>'
  [train] 'assistant'
  [train] 'Ċ'
  [MASK ] '<think>'
  [MASK ] 'ĊĊ'
  [MASK ] '</think>'
  [MASK ] 'ĊĊ'
  [train] '4'
  [train] '<|im_end|>'

=== OLMo-3 — think masked (MaskType.THINK) ===
  total tokens: 87, masked: 6, trainable: 81
  [train] '<|im_start|>'
  [train] 'system'
  [train] 'Ċ'
  [train] 'You'
  [train] 'Ġare'
  [train] 'ĠO'
  [train] 'LM'
  [train] 'o'
  [train] ','
  [train] 'Ġa'
  [train] 'Ġhelpful'
  [train] 'Ġfunction'
  [train] '-c'
  [train] 'alling'
  [train] 'ĠAI'
  [train] 'Ġassistant'
  [train] 'Ġbuilt'
  [train] 'Ġby'
  [train] 'ĠAi'
  [train] '2'
  [train] '.'
  [train] 'ĠYour'
  [train] 'Ġdate'
  [train] 'Ġcutoff'
  [train] 'Ġis'
  [train] 'ĠNovember

### 2c. Mask instruction and think block (`masked=MaskType.PROMPT | MaskType.THINK`)

A stricter variant: masks everything before the response. Only response tokens contribute to the loss. The think block is still injected and present as context — the response is always predicted from `[instruction + think block]`, matching the inference context.

In [6]:
# masked=MaskType.PROMPT | MaskType.THINK — only response tokens are trainable
for name, tok in [("Qwen3", tok_qwen), ("OLMo-3", tok_olmo)]:
    ds = apply_mask(
        conversations=[demo_conversation],
        tokenizer=tok,
        masked=MaskType.PROMPT | MaskType.THINK,
    )
    show_token_labels(ds, tok, f"{name} — prompt+think masked (MaskType.PROMPT | MaskType.THINK)")

=== Qwen3 — prompt+think masked (MaskType.PROMPT | MaskType.THINK) ===
  total tokens: 21, masked: 19, trainable: 2
  [MASK ] '<|im_start|>'
  [MASK ] 'user'
  [MASK ] 'Ċ'
  [MASK ] 'What'
  [MASK ] 'Ġis'
  [MASK ] 'Ġ'
  [MASK ] '2'
  [MASK ] '+'
  [MASK ] '2'
  [MASK ] '?'
  [MASK ] '<|im_end|>'
  [MASK ] 'Ċ'
  [MASK ] '<|im_start|>'
  [MASK ] 'assistant'
  [MASK ] 'Ċ'
  [MASK ] '<think>'
  [MASK ] 'ĊĊ'
  [MASK ] '</think>'
  [MASK ] 'ĊĊ'
  [train] '4'
  [train] '<|im_end|>'

=== OLMo-3 — prompt+think masked (MaskType.PROMPT | MaskType.THINK) ===
  total tokens: 87, masked: 85, trainable: 2
  [MASK ] '<|im_start|>'
  [MASK ] 'system'
  [MASK ] 'Ċ'
  [MASK ] 'You'
  [MASK ] 'Ġare'
  [MASK ] 'ĠO'
  [MASK ] 'LM'
  [MASK ] 'o'
  [MASK ] ','
  [MASK ] 'Ġa'
  [MASK ] 'Ġhelpful'
  [MASK ] 'Ġfunction'
  [MASK ] '-c'
  [MASK ] 'alling'
  [MASK ] 'ĠAI'
  [MASK ] 'Ġassistant'
  [MASK ] 'Ġbuilt'
  [MASK ] 'Ġby'
  [MASK ] 'ĠAi'
  [MASK ] '2'
  [MASK ] '.'
  [MASK ] 'ĠYour'
  [MASK ] 'Ġdate'
  [MAS

## 3. Conversations With Reasoning Content

When the assistant message has a `"reasoning"` key with actual content, `MaskType.THINK` masks the full reasoning block (including tags) while the response remains trainable. This is the intended use case for distillation data — training on responses derived from chain-of-thought without propagating gradients through the reasoning.

Use `to_conversations()` from `thinkpack.distill` to convert distillation records directly into this format.

In [7]:
# conversation with actual reasoning content (e.g. distilled from a teacher model)
reasoning_conversation = [
    {"role": "user", "content": "What is 2+2?"},
    {
        "role": "assistant",
        "content": "4",
        "reasoning": "I need to add 2 and 2 together. 2 + 2 = 4.",
    },
]

for name, tok in [("Qwen3", tok_qwen), ("OLMo-3", tok_olmo)]:
    ds = apply_mask(
        conversations=[reasoning_conversation],
        tokenizer=tok,
        masked=MaskType.THINK,
    )
    show_token_labels(ds, tok, f"{name} — real reasoning, think masked")

=== Qwen3 — real reasoning, think masked ===
  total tokens: 41, masked: 24, trainable: 17
  [train] '<|im_start|>'
  [train] 'user'
  [train] 'Ċ'
  [train] 'What'
  [train] 'Ġis'
  [train] 'Ġ'
  [train] '2'
  [train] '+'
  [train] '2'
  [train] '?'
  [train] '<|im_end|>'
  [train] 'Ċ'
  [train] '<|im_start|>'
  [train] 'assistant'
  [train] 'Ċ'
  [MASK ] '<think>'
  [MASK ] 'Ċ'
  [MASK ] 'I'
  [MASK ] 'Ġneed'
  [MASK ] 'Ġto'
  [MASK ] 'Ġadd'
  [MASK ] 'Ġ'
  [MASK ] '2'
  [MASK ] 'Ġand'
  [MASK ] 'Ġ'
  [MASK ] '2'
  [MASK ] 'Ġtogether'
  [MASK ] '.'
  [MASK ] 'Ġ'
  [MASK ] '2'
  [MASK ] 'Ġ+'
  [MASK ] 'Ġ'
  [MASK ] '2'
  [MASK ] 'Ġ='
  [MASK ] 'Ġ'
  [MASK ] '4'
  [MASK ] '.Ċ'
  [MASK ] '</think>'
  [MASK ] 'ĊĊ'
  [train] '4'
  [train] '<|im_end|>'

=== OLMo-3 — real reasoning, think masked ===
  total tokens: 107, masked: 26, trainable: 81
  [train] '<|im_start|>'
  [train] 'system'
  [train] 'Ċ'
  [train] 'You'
  [train] 'Ġare'
  [train] 'ĠO'
  [train] 'LM'
  [train] 'o'
  [train] ','

## 4. Summary

| Model | `masked` | Think block in sequence? | Instruction trained? | Think tokens trained? | Response trained? |
|---|---|---|---|---|---|
| Qwen3 | `None` | Yes (template auto-adds) | Yes | Yes | Yes |
| Qwen3 | `MaskType.THINK` | Yes (template auto-adds) | Yes | **No** | Yes |
| Qwen3 | `PROMPT \| THINK` | Yes (template auto-adds) | **No** | **No** | Yes |
| OLMo-3 | `None` | No | Yes | N/A | Yes |
| OLMo-3 | `MaskType.THINK` | Yes (injected blank) | Yes | **No** | Yes |
| OLMo-3 | `PROMPT \| THINK` | Yes (injected blank) | **No** | **No** | Yes |

**Key points:**
- `apply_mask()` accepts conversations — lists of `{"role": ..., "content": ...}` dicts, matching `apply_chat_template()`. The assistant message may include a `"reasoning"` key.
- When masking is active, an empty `"reasoning"` field is automatically injected for conversations that lack one, keeping training context consistent with inference.
- Token boundaries are found by tokenizing text prefixes, which correctly handles models like Qwen3 that always emit a think block regardless of reasoning content.
- `MaskType` flags can be combined freely — including `MaskType.RESPONSE` to mask only the response (unusual, but valid).